# NB14 — Neural-only production model (the missing matched baseline)

NB10 gave us the **hybrid** trained on all 6 generators. For the stress test we need its **matched
neural-only twin**: same encoder, same K=9 chunking, same standardizer policy, same hyperparameters,
same data — the ONLY difference is that the statistical block is absent (fused dim 768 instead of 773).

Without this, a polishing/humanizing collapse curve has no reference line and says nothing about whether
fusion helps. The NB11 checkpoints cannot serve: each is missing one generator by construction.

Everything is printed to the log (no tqdm, no display). Checkpoint/resume every 200 steps and每 epoch.

## 1 · Config — identical to NB10 except USE_STAT=False

In [1]:
import os, time, random, numpy as np, pandas as pd, torch
P_DATASET  = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"          # EDIT
CKPT_DIR   = "/kaggle/working/ckpt_neural"
RESUME_FROM= "/kaggle/input/nb14-ckpt/ckpt_neural"                 # EDIT after a crash; ignored if absent
# reuse NB10's tokenised chunks if you promoted them (skips ~1 min of work):
P_CHUNKS   = "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/chunks_K9.npz"
W_CHUNKS   = "/kaggle/working/chunks_K9.npz"

MODEL_ID = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
USE_STAT = False                    # <<< the only difference from NB10
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT = 256, 0.0
LR_ENC, LR_HEAD, WD = 2e-5, 1e-3, 0.01
EPOCHS, MICRO_BS, GRAD_ACCUM, SAVE_EVERY = 3, 2, 16, 200
SEED = 42
SMOKE = False                       # True = 40-article dry run

os.makedirs(CKPT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[1/8] config loaded | device={DEV} | USE_STAT={USE_STAT} | K={K_CHUNKS} | SMOKE={SMOKE}", flush=True)
print(f"[1/8] epochs={EPOCHS} micro_bs={MICRO_BS} grad_accum={GRAD_ACCUM} "
      f"lr_enc={LR_ENC} lr_head={LR_HEAD}", flush=True)

[1/8] config loaded | device=cuda | USE_STAT=False | K=9 | SMOKE=False
[1/8] epochs=3 micro_bs=2 grad_accum=16 lr_enc=2e-05 lr_head=0.001


## 2 · Load data

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
y     = df["label"].to_numpy(np.int64)
split = df["split"].to_numpy()
texts = df["text"].astype(str).tolist()
print(f"[2/8] dataset: {len(df)} articles | human {int((y==0).sum())} | AI {int((y==1).sum())}", flush=True)
for s in ("train","val","test"):
    print(f"[2/8]   {s}: {int((split==s).sum())}", flush=True)

[2/8] dataset: 7101 articles | human 3500 | AI 3601
[2/8]   train: 5363
[2/8]   val: 645
[2/8]   test: 1093


## 3 · Chunk ids (load cached, else build)

In [3]:
!pip install -q transformers
from transformers import AutoTokenizer
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
CHUNK_LEN = MAX_CT + 2
print(f"[3/8] tokenizer loaded ({time.time()-t0:.1f}s)", flush=True)

def to_chunks(t):
    ids = tok(t, add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
    wins = [ids[i:i+MAX_CT] for i in range(0, len(ids), STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
    return [[CLS]+w+[SEP]+[PAD]*(CHUNK_LEN-2-len(w)) for w in wins]

src = next((p for p in (W_CHUNKS, P_CHUNKS) if p and os.path.exists(p)), None)
if src:
    z = np.load(src, allow_pickle=True); CH, NCH = z["ch"], z["nch"]
    print(f"[3/8] loaded cached chunks from {src}", flush=True)
else:
    t0 = time.time()
    CH = np.full((len(texts), K_CHUNKS, CHUNK_LEN), PAD, np.int32); NCH = np.zeros(len(texts), np.int8)
    for i, t in enumerate(texts):
        ch = to_chunks(t); NCH[i] = len(ch)
        for j, c in enumerate(ch): CH[i, j] = c
        if (i+1) % 1000 == 0: print(f"[3/8]   chunked {i+1}/{len(texts)}", flush=True)
    np.savez_compressed(W_CHUNKS, ch=CH, nch=NCH)
    print(f"[3/8] built + saved chunks in {time.time()-t0:.1f}s -> {W_CHUNKS}", flush=True)
assert CH.shape[0] == len(df)
print(f"[3/8] chunk tensor {CH.shape} | mean chunks/article {NCH.mean():.2f} | max {NCH.max()}", flush=True)

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[3/8] tokenizer loaded (0.4s)
[3/8] loaded cached chunks from /kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/chunks_K9.npz
[3/8] chunk tensor (7101, 9, 512) | mean chunks/article 2.68 | max 9


## 4 · Model (neural-only) + dataset

In [4]:
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import DataLoader, Dataset

class Net(nn.Module):
    def __init__(self, use_stat=False, stat_dim=0):
        super().__init__()
        self.use_stat = use_stat
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        self.enc.gradient_checkpointing_enable()
        h = self.enc.config.hidden_size
        fused = h + (stat_dim if use_stat else 0)
        self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
        self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
        self.fused_dim = fused
    def encode(self, ids, nch):
        B,K,L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
        cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:,0,:].view(B,K,-1)
        m = (torch.arange(K, device=ids.device)[None,:] < nch[:,None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, ids, nch, stat=None):
        v = self.encode(ids, nch)
        v = (v - self.mu)/self.sd
        return self.head(v)

class DS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        i = self.rows[k]
        return torch.from_numpy(CH[i].astype(np.int64)), int(NCH[i]), int(y[i]), i
def collate(b):
    return (torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]),
            torch.tensor([x[2] for x in b]), [x[3] for x in b])

train_rows = [i for i in range(len(df)) if split[i] == "train"]
if SMOKE:
    a = [i for i in train_rows if y[i]==0][:20]; b = [i for i in train_rows if y[i]==1][:20]
    train_rows = a + b
print(f"[4/8] model defined | train rows {len(train_rows)}", flush=True)

[4/8] model defined | train rows 5363


## 5 · Standardizer from TRAIN only (768-dim, one frozen pass)

In [5]:
@torch.no_grad()
def fit_std(model, rows):
    model.eval(); s = s2 = None; n = 0; t0 = time.time()
    dl = DataLoader(DS(rows), batch_size=8, collate_fn=collate)
    for bi,(ids,nch,yy,idx) in enumerate(dl, 1):
        ids, nch = ids.to(DEV), nch.to(DEV)
        v = model.encode(ids, nch)
        s  = v.sum(0) if s is None else s + v.sum(0)
        s2 = (v*v).sum(0) if s2 is None else s2 + (v*v).sum(0)
        n += v.shape[0]
        if bi % 50 == 0: print(f"[5/8]   standardizer {n}/{len(rows)} ({time.time()-t0:.0f}s)", flush=True)
    mu = s/n; sd = torch.sqrt(torch.clamp(s2/n - mu*mu, min=1e-6))
    model.mu.copy_(mu); model.sd.copy_(sd)
    print(f"[5/8] standardizer fitted on {n} train articles in {time.time()-t0:.0f}s "
          f"| mu|.|={float(mu.abs().mean()):.4f} sd_mean={float(sd.mean()):.4f}", flush=True)
print("[5/8] routine ready", flush=True)

[5/8] routine ready


## 6 · Checkpoint helpers

In [6]:
def save_ck(model, opt, sched, scaler, ep, step, name="last.pt"):
    p = os.path.join(CKPT_DIR, name)
    torch.save({"model":model.state_dict(), "opt":opt.state_dict(), "sched":sched.state_dict(),
                "scaler":scaler.state_dict(), "ep":ep, "step":step,
                "trng":torch.get_rng_state(), "crng":torch.cuda.get_rng_state_all(),
                "nrng":np.random.get_state(), "prng":random.getstate()}, p)
    print(f"[6/8]   checkpoint saved -> {name} (epoch {ep}, step {step})", flush=True)

def find_resume():
    for base in (CKPT_DIR, RESUME_FROM):
        p = os.path.join(base, "last.pt")
        if os.path.exists(p): return p
    return None
print("[6/8] checkpoint helpers ready", flush=True)

[6/8] checkpoint helpers ready


## 7 · Train

In [7]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

model = Net(use_stat=USE_STAT, stat_dim=0).to(DEV)
print(f"[7/8] model on {DEV} | fused dim {model.fused_dim} | "
      f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M", flush=True)

cw = compute_class_weight("balanced", classes=np.array([0,1]), y=y[train_rows])
lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEV))
print(f"[7/8] class weights {cw.round(4).tolist()}", flush=True)

dl = DataLoader(DS(train_rows), batch_size=MICRO_BS, shuffle=True, collate_fn=collate, drop_last=True)
total = (len(dl)//GRAD_ACCUM)*EPOCHS
enc_p = [p for n,p in model.named_parameters() if n.startswith("enc.")]
hd_p  = [p for n,p in model.named_parameters() if not n.startswith("enc.")]
opt = torch.optim.AdamW([{"params":enc_p,"lr":LR_ENC},{"params":hd_p,"lr":LR_HEAD}], weight_decay=WD)
sched = get_linear_schedule_with_warmup(opt, int(0.06*total), total)
scaler = torch.amp.GradScaler('cuda')
print(f"[7/8] batches/epoch {len(dl)} | optimizer steps total {total}", flush=True)

rp = find_resume(); start_ep = gstep = 0
if rp:
    ck = torch.load(rp, map_location=DEV, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    torch.set_rng_state(ck["trng"].cpu()); torch.cuda.set_rng_state_all([s.cpu() for s in ck["crng"]])
    np.random.set_state(ck["nrng"]); random.setstate(ck["prng"])
    start_ep, gstep = ck["ep"], ck["step"]
    print(f"[7/8] RESUMED from {rp}: epoch {start_ep}, step {gstep}", flush=True)
else:
    print("[7/8] fresh start -> fitting standardizer", flush=True)
    fit_std(model, train_rows)
    save_ck(model, opt, sched, scaler, 0, 0)

print("[7/8] training starts", flush=True); print("-"*90, flush=True)
T0 = time.time(); model.train()
for ep in range(start_ep, EPOCHS):
    ep_t0 = time.time(); run_loss = 0.0; nb = 0; opt.zero_grad()
    for bi,(ids,nch,yy,idx) in enumerate(dl):
        ids, nch, yy = ids.to(DEV), nch.to(DEV), yy.to(DEV)
        with torch.amp.autocast('cuda'):
            loss = lossf(model(ids, nch), yy)/GRAD_ACCUM
        scaler.scale(loss).backward()
        run_loss += loss.item()*GRAD_ACCUM; nb += 1
        if (bi+1) % GRAD_ACCUM == 0:
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(); gstep += 1
            if gstep % 20 == 0:
                el = time.time()-T0
                eta = (total-gstep)/max(gstep/max(el,1e-9),1e-9)/60
                print(f"[7/8] ep{ep+1}/{EPOCHS} step {gstep}/{total} "
                      f"loss {run_loss/max(nb,1):.4f} lr {sched.get_last_lr()[0]:.2e} "
                      f"| {el/60:.1f}m elapsed, ETA {eta:.0f}m", flush=True)
                run_loss = 0.0; nb = 0
            if gstep % SAVE_EVERY == 0:
                save_ck(model, opt, sched, scaler, ep, gstep)
    save_ck(model, opt, sched, scaler, ep+1, gstep, f"epoch{ep+1}.pt")
    save_ck(model, opt, sched, scaler, ep+1, gstep)
    print(f"[7/8] EPOCH {ep+1} done in {(time.time()-ep_t0)/60:.1f} min", flush=True)
print("-"*90, flush=True)
print(f"[7/8] TRAINING COMPLETE in {(time.time()-T0)/60:.1f} min", flush=True)

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[7/8] model on cuda | fused dim 768 | params 109.3M
[7/8] class weights [1.0146, 0.9858]
[7/8] batches/epoch 2681 | optimizer steps total 501
[7/8] fresh start -> fitting standardizer
[5/8]   standardizer 400/5363 (95s)
[5/8]   standardizer 800/5363 (204s)
[5/8]   standardizer 1200/5363 (313s)
[5/8]   standardizer 1600/5363 (421s)
[5/8]   standardizer 2000/5363 (530s)
[5/8]   standardizer 2400/5363 (638s)
[5/8]   standardizer 2800/5363 (747s)
[5/8]   standardizer 3200/5363 (856s)
[5/8]   standardizer 3600/5363 (964s)
[5/8]   standardizer 4000/5363 (1072s)
[5/8]   standardizer 4400/5363 (1181s)
[5/8]   standardizer 4800/5363 (1289s)
[5/8]   standardizer 5200/5363 (1398s)
[5/8] standardizer fitted on 5363 train articles in 1444s | mu|.|=0.4760 sd_mean=0.3682
[6/8]   checkpoint saved -> last.pt (epoch 0, step 0)
[7/8] training starts
------------------------------------------------------------------------------------------


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


[7/8] ep1/3 step 20/501 loss 0.3917 lr 1.33e-05 | 3.0m elapsed, ETA 72m
[7/8] ep1/3 step 40/501 loss 0.1056 lr 1.96e-05 | 6.0m elapsed, ETA 69m
[7/8] ep1/3 step 60/501 loss 0.0472 lr 1.87e-05 | 9.0m elapsed, ETA 66m
[7/8] ep1/3 step 80/501 loss 0.0211 lr 1.79e-05 | 12.0m elapsed, ETA 63m
[7/8] ep1/3 step 100/501 loss 0.0223 lr 1.70e-05 | 15.0m elapsed, ETA 60m
[7/8] ep1/3 step 120/501 loss 0.0216 lr 1.62e-05 | 17.9m elapsed, ETA 57m
[7/8] ep1/3 step 140/501 loss 0.0147 lr 1.53e-05 | 20.9m elapsed, ETA 54m
[7/8] ep1/3 step 160/501 loss 0.0562 lr 1.45e-05 | 23.9m elapsed, ETA 51m
[6/8]   checkpoint saved -> epoch1.pt (epoch 1, step 167)
[6/8]   checkpoint saved -> last.pt (epoch 1, step 167)
[7/8] EPOCH 1 done in 25.1 min
[7/8] ep2/3 step 180/501 loss 0.0163 lr 1.36e-05 | 27.1m elapsed, ETA 48m
[7/8] ep2/3 step 200/501 loss 0.0036 lr 1.28e-05 | 30.1m elapsed, ETA 45m
[6/8]   checkpoint saved -> last.pt (epoch 1, step 200)
[7/8] ep2/3 step 220/501 loss 0.0033 lr 1.19e-05 | 33.1m elapsed, 

## 8 · Evaluate on the test split

In [8]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, confusion_matrix

@torch.no_grad()
def predict(rows):
    model.eval(); P=[]; Y=[]; t0=time.time()
    dl_ = DataLoader(DS(rows), batch_size=8, collate_fn=collate)
    for bi,(ids,nch,yy,idx) in enumerate(dl_, 1):
        ids, nch = ids.to(DEV), nch.to(DEV)
        with torch.amp.autocast('cuda'):
            p = torch.softmax(model(ids, nch), 1)[:,1]
        P.append(p.float().cpu().numpy()); Y.append(yy.numpy())
        if bi % 20 == 0: print(f"[8/8]   predicted {bi*8}/{len(rows)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(P), np.concatenate(Y)

test_rows = [i for i in range(len(df)) if split[i] == "test"]
print(f"[8/8] evaluating on {len(test_rows)} test articles", flush=True)
p, yt = predict(test_rows); pred = (p >= .5).astype(int)
tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
print(f"[8/8] MacroF1 {100*f1_score(yt,pred,average='macro'):.2f} | "
      f"Acc {100*accuracy_score(yt,pred):.2f} | AUC {100*roc_auc_score(yt,p):.2f}", flush=True)
print(f"[8/8] confusion: TN {tn} FP {fp} FN {fn} TP {tp} | "
      f"FPR {100*fp/max(tn+fp,1):.2f}% | TPR {100*tp/max(tp+fn,1):.2f}%", flush=True)
np.savez("/kaggle/working/nb14_test_preds.npz", p=p, y=yt,
         article_id=np.array([df.index[i] for i in test_rows]))
print("[8/8] saved test predictions -> /kaggle/working/nb14_test_preds.npz", flush=True)
print("[8/8] NOTE: compare against NB10 hybrid (std MacroF1 99.82) — this is its matched twin.", flush=True)

[8/8] evaluating on 1093 test articles
[8/8]   predicted 160/1093 (11s)
[8/8]   predicted 320/1093 (22s)
[8/8]   predicted 480/1093 (34s)
[8/8]   predicted 640/1093 (45s)
[8/8]   predicted 800/1093 (56s)
[8/8]   predicted 960/1093 (67s)
[8/8] MacroF1 99.91 | Acc 99.91 | AUC 100.00
[8/8] confusion: TN 538 FP 1 FN 0 TP 554 | FPR 0.19% | TPR 100.00%
[8/8] saved test predictions -> /kaggle/working/nb14_test_preds.npz
[8/8] NOTE: compare against NB10 hybrid (std MacroF1 99.82) — this is its matched twin.
